# Day 3 - Create Batches for Transforming Historical Price with Summary

In [23]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from openai import OpenAI
from litellm import completion
from tqdm.notebook import tqdm
from IPython.display import display, Markdown
from pricer.evaluator import evaluate

In [24]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [25]:
openai = OpenAI(api_key=OPENAI_API_KEY)

In [26]:
MODEL = "gpt-4.1-nano"

In [27]:
processed_df = pd.read_csv("dfs/processed_stock_data.csv")

In [28]:
processed_df

,Id,Ticker,Start Date,End Date,Last Price,Price History,Future Price,Return %
0,0,ADMR.JK,2022-10-27,2022-12-07,1572.644165,"Date: 2022-10-27 00:00:00, Close: 1667.96, Vol...",1620.30,3.03
1,1,ADMR.JK,2022-10-28,2022-12-08,1601.237671,"Date: 2022-10-28 00:00:00, Close: 1644.13, Vol...",1644.13,2.68
2,2,ADMR.JK,2022-10-31,2022-12-09,1591.706543,"Date: 2022-10-31 00:00:00, Close: 1648.89, Vol...",1687.02,5.99
3,3,ADMR.JK,2022-11-01,2022-12-12,1596.472046,"Date: 2022-11-01 00:00:00, Close: 1606.00, Vol...",1667.96,4.48
4,4,ADMR.JK,2022-11-02,2022-12-13,1596.472046,"Date: 2022-11-02 00:00:00, Close: 1667.96, Vol...",1672.72,4.78
...,...,...,...,...,...,...,...,...
13331,13331,BUMI.JK,2026-01-13,2026-02-26,258.000000,"Date: 2026-01-13 00:00:00, Close: 406.00, Volu...",248.00,-3.88
13332,13332,BUMI.JK,2026-01-14,2026-02-27,258.000000,"Date: 2026-01-14 00:00:00, Close: 422.00, Volu...",242.00,-6.20
13333,13333,BUMI.JK,2026-01-15,2026-03-02,248.000000,"Date: 2026-01-15 00:00:00, Close: 410.00, Volu...",240.00,-3.23
13334,13334,BUMI.JK,2026-01-19,2026-03-03,252.000000,"Date: 2026-01-19 00:00:00, Close: 412.00, Volu...",240.00,-4.76


In [29]:
train = processed_df.iloc[:int(len(processed_df)*0.8)].reset_index(drop=True)
val = processed_df.iloc[int(len(processed_df)*0.8):int(len(processed_df)*0.9)].reset_index(drop=True)
test = processed_df.iloc[int(len(processed_df)*0.9):].reset_index(drop=True)

In [30]:
print(f"Train size: {len(train)}, Val size: {len(val)}, Test size: {len(test)}")

Train size: 10668, Val size: 1334, Test size: 1334


In [10]:
SYSTEM_PROMPT = """
You are a quantitative financial analyst. You will receive 30 days of historical stock data including:
closing price, volume, MA-200, MA-150, MA-50, 10-day momentum, and 14-day RSI.

Your job is to produce a structured summary that captures the information most predictive of price direction over the next 30 days. Be concise and signal-focused — avoid restating raw numbers unless they mark a key level.

Output your summary in this exact format:

TREND: [Bullish / Bearish / Neutral] — one sentence on the dominant price trend.
MOMENTUM: One sentence on momentum and RSI trajectory (accelerating, fading, reversing).
MA_STRUCTURE: One sentence on price position relative to MA-50, MA-150, MA-200 (above/below, converging/diverging).
VOLUME: One sentence on notable volume events and what they suggest about conviction.
KEY_LEVELS: Comma-separated support and resistance levels inferred from the data.
RISK_FACTORS: One sentence on the biggest uncertainty or warning sign in the data.
PREDICTION_CONTEXT: One sentence synthesizing the above into the most likely price regime over the next 30 days (directional bias only, no specific price target).
"""

In [ ]:
SYSTEM_PROMPT = """
You are an expert financial analyst assistant. Your task is to analyze historical stock price data and make a concise summary of the key trends and patterns to predict future performance. Attach a detailed analysis of the price movements and technical indicators. Do not include key takeaways or conclusions. You will be given a price history of a stock for the past 30 days, including the closing price, trading volume, moving averages (MA-200, MA-150, MA-50), momentum (10-day), and RSI (14-day).
"""

In [11]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": processed_df.iloc[0]["Price History"]},
]

response = completion(model="gpt-4.1-nano", messages=messages)

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 2635
Output tokens: 187
Total cost: 0.0338 cents


In [12]:
display(Markdown(f"**Model Response:**\n\n{response.choices[0].message.content}"))

**Model Response:**

TREND: Bearish — the stock has been declining steadily, trading below all key moving averages with no clear reversal in sight.  
MOMENTUM: Momentum has weakened significantly, with RSI staying below 50 and a recent decline in momentum indicating fading buying interest.  
MA_STRUCTURE: The price remains below the long-term MA-200, and shorter-term MAs are flat or declining, showing no signs of trend reversal.  
VOLUME: Large volume spikes during down days suggest high conviction in the decline but lack of sustained buying support.  
KEY_LEVELS: Support around 1550, resistance at approximately 1650.  
RISK_FACTORS: The recent oversold RSI near 23-35 indicates potential for a short-term oversold bounce, but overall trend remains downward.  
PREDICTION_CONTEXT: The dominant bearish momentum and price below key MAs point to continued downside over the next 30 days.

In [13]:
def make_jsonl(row):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": row["Price History"]}]}
    line = {"custom_id": str(row["Id"]), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [14]:
make_jsonl(processed_df.iloc[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4.1-nano", "messages": [{"role": "system", "content": "\\nYou are a quantitative financial analyst. You will receive 30 days of historical stock data including:\\nclosing price, volume, MA-200, MA-150, MA-50, 10-day momentum, and 14-day RSI.\\n\\nYour job is to produce a structured summary that captures the information most predictive of price direction over the next 30 days. Be concise and signal-focused \\u2014 avoid restating raw numbers unless they mark a key level.\\n\\nOutput your summary in this exact format:\\n\\nTREND: [Bullish / Bearish / Neutral] \\u2014 one sentence on the dominant price trend.\\nMOMENTUM: One sentence on momentum and RSI trajectory (accelerating, fading, reversing).\\nMA_STRUCTURE: One sentence on price position relative to MA-50, MA-150, MA-200 (above/below, converging/diverging).\\nVOLUME: One sentence on notable volume events and what they suggest about convictio

In [15]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            jsonl = make_jsonl(processed_df.iloc[i])
            f.write(jsonl + "\n")

In [ ]:
make_file(13000, 13336, "jsonl/batches/13000_13336.jsonl")

In [ ]:
with open("jsonl/batches/13000_13336.jsonl", "rb") as f:
    response = openai.files.create(file=f, purpose="batch")
response

FileObject(id='file-YbxioGsrvfcQKxnBa4se2p', bytes=1895439, created_at=1777125669, filename='13000_13336.jsonl', object='file', purpose='batch', status='processed', expires_at=1779717669, status_details=None)

In [191]:
batch = openai.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=response.id)

In [192]:
result = openai.batches.retrieve(batch.id)

In [193]:
result

Batch(id='batch_69ecc926c6408190bd876005e12733d5', completion_window='24h', created_at=1777125670, endpoint='/v1/chat/completions', input_file_id='file-YbxioGsrvfcQKxnBa4se2p', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777125836, error_file_id=None, errors=None, expired_at=None, expires_at=1777212070, failed_at=None, finalizing_at=1777125802, in_progress_at=1777125671, metadata=None, model='gpt-4.1-nano-2025-04-14', output_file_id='file-7aWEMNUqouj5WNK5uVfb9b', request_counts=BatchRequestCounts(completed=336, failed=0, total=336), usage=BatchUsage(input_tokens=851448, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=71323, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=922771))

In [ ]:
response = openai.files.content(result.output_file_id)
response.write_to_file("jsonl/output/13000_13336.jsonl")

In [31]:
for start in tqdm(range(0, len(processed_df), 500)):
    end = min(start + 500, len(processed_df))
    with open(f"jsonl/output/{start}_{end}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            json_line = json.loads(line)
            id = int(json_line["custom_id"])
            summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
            processed_df.loc[processed_df["Id"] == id, "Price History"] = summary
        f.close()

  0%|          | 0/27 [00:00<?, ?it/s]

In [32]:
print(f"5 sample summaries:\n")
for i in range(5):
    print(processed_df.iloc[i]["Price History"])

5 sample summaries:

TREND: Bearish — the stock has been declining sharply, with momentum and price below all key moving averages, indicating a dominant downtrend.  
MOMENTUM: Momentum has been fading and remains negative with RSI in oversold territory, suggesting weakening bullish attempts and potential short-term stabilization.  
MA_STRUCTURE: Price is below the 50, 150, and 200-day MAs, which are all diverging upwards but remain well above the current price, confirming downward pressure.  
VOLUME: Volume spikes on recent declines indicate high conviction in continued selling; volume during rebounds is comparatively lower, further emphasizing bearish sentiment.  
KEY_LEVELS: Support around 1550, Resistance near 1710, with key moving averages acting as dynamic resistance levels.  
RISK_FACTORS: The persistent oversold RSI and volatility may lead to short-term bounces, but the overall trend remains uncertain without clear reversal signals.  
PREDICTION_CONTEXT: The dominant trend and m

In [33]:
processed_df.to_csv("dfs/processed_stock_data_with_summaries.csv", index=False)

In [34]:
processed_train = processed_df.iloc[:int(0.8 * len(processed_df))].reset_index(drop=True)
processed_val = processed_df.iloc[int(0.8 * len(processed_df)):int(0.9 * len(processed_df))].reset_index(drop=True)
processed_test = processed_df.iloc[int(0.9 * len(processed_df)):].reset_index(drop=True)

In [35]:
DatasetDict({
    'train': Dataset.from_pandas(processed_train),
    'validation': Dataset.from_pandas(processed_val),
    'test': Dataset.from_pandas(processed_test)
}).push_to_hub("matthewyn/stocks")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/matthewyn/stocks/commit/0b391e7d858817148e1647d331e291b771f4ca4f', commit_message='Upload dataset', commit_description='', oid='0b391e7d858817148e1647d331e291b771f4ca4f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/matthewyn/stocks', endpoint='https://huggingface.co', repo_type='dataset', repo_id='matthewyn/stocks'), pr_revision=None, pr_num=None)